# Walk of Life 2026 – Runner Photo Recognizer

This notebook uses a **pretrained PyTorch model** (ResNet-18) fine-tuned to
identify runner photos from the *Walk of Life 2026* event in Naples.

**Requirements**
- Google Colab with GPU runtime (Runtime → Change runtime type → T4 GPU)
- Dataset stored on Google Drive in the structure described below

## Expected Dataset Structure on Google Drive

```
My Drive/
  walk_of_life_dataset/
    train/
      runner/
        img001.jpg
        img002.jpg
        ...
      non_runner/
        img001.jpg
        img002.jpg
        ...
    val/
      runner/
        img001.jpg
        ...
      non_runner/
        img001.jpg
        ...
```

Each class folder contains the images for that category.
You can change the `DATASET_ROOT` variable below to match your path.

## 1 – Setup & Google Drive Mount

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# PyTorch and torchvision are pre-installed on Colab.
# Uncomment the line below only if you need a specific version.
# !pip install torch torchvision

In [ ]:
import os
import copy
import time

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader

import torchvision
from torchvision import datasets, models, transforms

import matplotlib.pyplot as plt
import numpy as np
from PIL import Image
from sklearn.metrics import classification_report, confusion_matrix

# Reproducibility
torch.manual_seed(42)
np.random.seed(42)

In [ ]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')
if device.type == 'cuda':
    print(f'GPU: {torch.cuda.get_device_name(0)}')

## 2 – Configuration

In [ ]:
# ---------- paths ----------
DATASET_ROOT = '/content/drive/MyDrive/walk_of_life_dataset'
MODEL_SAVE_PATH = '/content/drive/MyDrive/walk_of_life_model.pth'

# ---------- hyper-parameters ----------
NUM_CLASSES = 2        # runner / non_runner
BATCH_SIZE = 32
NUM_EPOCHS = 10
LEARNING_RATE = 1e-4
IMAGE_SIZE = 224       # ResNet expected input size

## 3 – Data Loading from Google Drive

In [ ]:
data_transforms = {
    'train': transforms.Compose([
        transforms.RandomResizedCrop(IMAGE_SIZE),
        transforms.RandomHorizontalFlip(),
        transforms.ColorJitter(brightness=0.2, contrast=0.2,
                               saturation=0.2, hue=0.1),
        transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406],
                             [0.229, 0.224, 0.225]),
    ]),
    'val': transforms.Compose([
        transforms.Resize(256),
        transforms.CenterCrop(IMAGE_SIZE),
        transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406],
                             [0.229, 0.224, 0.225]),
    ]),
}

In [ ]:
image_datasets = {
    split: datasets.ImageFolder(
        os.path.join(DATASET_ROOT, split),
        data_transforms[split],
    )
    for split in ['train', 'val']
}

dataloaders = {
    split: DataLoader(
        image_datasets[split],
        batch_size=BATCH_SIZE,
        shuffle=(split == 'train'),
        num_workers=2,
    )
    for split in ['train', 'val']
}

dataset_sizes = {split: len(image_datasets[split]) for split in ['train', 'val']}
class_names = image_datasets['train'].classes

print(f'Classes: {class_names}')
print(f'Training images: {dataset_sizes["train"]}')
print(f'Validation images: {dataset_sizes["val"]}')

In [ ]:
def imshow(inp, title=None):
    """Display a tensor as an image (undo normalisation)."""
    inp = inp.numpy().transpose((1, 2, 0))
    mean = np.array([0.485, 0.456, 0.406])
    std = np.array([0.229, 0.224, 0.225])
    inp = std * inp + mean
    inp = np.clip(inp, 0, 1)
    plt.imshow(inp)
    if title is not None:
        plt.title(title)
    plt.axis('off')

# Show a batch of training images
inputs, classes = next(iter(dataloaders['train']))
grid = torchvision.utils.make_grid(inputs[:8], nrow=4)
imshow(grid, title=[class_names[c] for c in classes[:8]])
plt.show()

## 4 – Pretrained Model (ResNet-18)

In [ ]:
def build_model(num_classes: int, freeze_backbone: bool = True):
    """Load a pretrained ResNet-18 and replace the final FC layer."""
    model = models.resnet18(weights=models.ResNet18_Weights.IMAGENET1K_V1)

    # Optionally freeze all backbone layers
    if freeze_backbone:
        for param in model.parameters():
            param.requires_grad = False

    # Replace the classification head
    num_features = model.fc.in_features
    model.fc = nn.Linear(num_features, num_classes)

    return model.to(device)

model = build_model(NUM_CLASSES, freeze_backbone=True)
print(model)

## 5 – Training

In [ ]:
def train_model(model, criterion, optimizer, scheduler, num_epochs):
    """Fine-tune the model and return the best weights."""
    best_model_wts = copy.deepcopy(model.state_dict())
    best_acc = 0.0
    history = {'train_loss': [], 'val_loss': [],
               'train_acc': [], 'val_acc': []}

    for epoch in range(num_epochs):
        print(f'Epoch {epoch + 1}/{num_epochs}')
        print('-' * 30)

        for phase in ['train', 'val']:
            if phase == 'train':
                model.train()
            else:
                model.eval()

            running_loss = 0.0
            running_corrects = 0

            for inputs, labels in dataloaders[phase]:
                inputs = inputs.to(device)
                labels = labels.to(device)

                optimizer.zero_grad()

                with torch.set_grad_enabled(phase == 'train'):
                    outputs = model(inputs)
                    _, preds = torch.max(outputs, 1)
                    loss = criterion(outputs, labels)

                    if phase == 'train':
                        loss.backward()
                        optimizer.step()

                running_loss += loss.item() * inputs.size(0)
                running_corrects += torch.sum(preds == labels.data)

            if phase == 'train' and scheduler is not None:
                scheduler.step()

            epoch_loss = running_loss / dataset_sizes[phase]
            epoch_acc = running_corrects.double() / dataset_sizes[phase]

            history[f'{phase}_loss'].append(epoch_loss)
            history[f'{phase}_acc'].append(epoch_acc.item())

            print(f'{phase:>5s}  Loss: {epoch_loss:.4f}  Acc: {epoch_acc:.4f}')

            if phase == 'val' and epoch_acc > best_acc:
                best_acc = epoch_acc
                best_model_wts = copy.deepcopy(model.state_dict())

        print()

    print(f'Best val accuracy: {best_acc:.4f}')
    model.load_state_dict(best_model_wts)
    return model, history

In [ ]:
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.fc.parameters(), lr=LEARNING_RATE)
scheduler = optim.lr_scheduler.StepLR(optimizer, step_size=5, gamma=0.1)

model, history = train_model(
    model, criterion, optimizer, scheduler, num_epochs=NUM_EPOCHS
)

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

ax1.plot(history['train_loss'], label='Train')
ax1.plot(history['val_loss'], label='Validation')
ax1.set_title('Loss')
ax1.set_xlabel('Epoch')
ax1.legend()

ax2.plot(history['train_acc'], label='Train')
ax2.plot(history['val_acc'], label='Validation')
ax2.set_title('Accuracy')
ax2.set_xlabel('Epoch')
ax2.legend()

plt.tight_layout()
plt.show()

## 6 – Save & Load Model

In [ ]:
torch.save(model.state_dict(), MODEL_SAVE_PATH)
print(f'Model saved to {MODEL_SAVE_PATH}')

In [ ]:
# To reload later:
# model = build_model(NUM_CLASSES, freeze_backbone=False)
# model.load_state_dict(torch.load(MODEL_SAVE_PATH, map_location=device))
# model.eval()

## 7 – Evaluation

In [ ]:
model.eval()
all_preds = []
all_labels = []

with torch.no_grad():
    for inputs, labels in dataloaders['val']:
        inputs = inputs.to(device)
        labels = labels.to(device)
        outputs = model(inputs)
        _, preds = torch.max(outputs, 1)
        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())


print(classification_report(all_labels, all_preds, target_names=class_names))
print('Confusion Matrix:')
print(confusion_matrix(all_labels, all_preds))

## 8 – Inference on a Single Image

In [ ]:
def predict_image(image_path: str, model, transform, class_names):
    """Predict the class of a single image and display it."""
    model.eval()
    image = Image.open(image_path).convert('RGB')
    input_tensor = transform(image).unsqueeze(0).to(device)

    with torch.no_grad():
        outputs = model(input_tensor)
        probabilities = torch.nn.functional.softmax(outputs, dim=1)
        confidence, predicted = torch.max(probabilities, 1)

    label = class_names[predicted.item()]
    conf = confidence.item() * 100

    plt.imshow(image)
    plt.title(f'{label} ({conf:.1f}%)')
    plt.axis('off')
    plt.show()

    return label, conf

In [ ]:
# Example: predict on a single image from Google Drive
# test_image = '/content/drive/MyDrive/walk_of_life_dataset/val/runner/img001.jpg'
# label, conf = predict_image(
#     test_image, model, data_transforms['val'], class_names
# )
# print(f'Prediction: {label} with {conf:.1f}% confidence')